# AI013 — Forecasting Training
**Phoenix Project | AI-ML Team**

Trains an LSTM model on the real wildfire dataset (Australia, 2024–2025).

> Place `wildfire_multi_region_dataset.csv` in the `data/` folder before running.

In [ ]:
# !pip install pandas numpy scikit-learn torch matplotlib seaborn pyyaml

In [ ]:
import os, sys
import yaml
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

sys.path.insert(0, os.path.join(os.getcwd(), '..', 'src'))

from dataset_builder import build_dataset, FEATURE_COLS, TARGET_COL, DATE_COL
from sequence_generator import prepare_sequences
from models.model import LSTMForecaster, fit
from evaluation.evaluate import set_seeds, plot_loss_curve, compute_metrics, print_metrics
from scoring.forecasting_scoring import score_model, score_baseline, save_checkpoint

with open('../configs/forecasting.yaml') as f:
    cfg = yaml.safe_load(f)

set_seeds(cfg['training']['random_seed'])
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'device: {device}')

## Step 1 — Load Dataset

In [ ]:
DATA_PATH = os.path.join('..', cfg['dataset']['path'])
df = build_dataset(DATA_PATH, region=cfg['dataset']['region_filter'])
print(df[FEATURE_COLS + [TARGET_COL]].describe().round(2))

In [ ]:
fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(df[DATE_COL], df[TARGET_COL], color='#c0392b', linewidth=1.2)
ax.set_title('Daily Average Fire Radiative Power — Australia', fontsize=13, fontweight='bold')
ax.set_xlabel('Date'); ax.set_ylabel('FRP (MW)')
plt.tight_layout(); plt.show()

## Step 2 — Create Sequences

In [ ]:
seq = prepare_sequences(
    df,
    feature_cols = FEATURE_COLS,
    target_col   = TARGET_COL,
    window       = cfg['sequence']['window_size'],
    train_split  = cfg['sequence']['train_split'],
    batch_size   = cfg['training']['batch_size']
)

X_train_t = seq['X_train_t']
y_train_t = seq['y_train_t']
X_test_t  = seq['X_test_t']
y_test_t  = seq['y_test_t']
train_loader   = seq['train_loader']
feat_scaler    = seq['feat_scaler']
target_scaler  = seq['target_scaler']
split          = seq['split_idx']

## Step 3 — Train Model

In [ ]:
model = LSTMForecaster(
    input_size  = len(FEATURE_COLS),
    hidden_size = cfg['model']['hidden_size'],
    dropout     = cfg['model']['dropout']
).to(device)

train_losses, val_losses = fit(
    model, train_loader, X_test_t, y_test_t,
    epochs       = cfg['training']['epochs'],
    lr           = cfg['training']['lr'],
    weight_decay = cfg['training']['weight_decay'],
    patience     = cfg['training']['lr_patience'],
    grad_clip    = cfg['training']['grad_clip'],
    device       = device
)

plot_loss_curve(train_losses, val_losses)

## Step 4 — Evaluate

In [ ]:
y_test_actual = target_scaler.inverse_transform(y_test_t.numpy())

lstm_metrics, lstm_preds = score_model(model, X_test_t, y_test_actual, target_scaler, device)
base_metrics, base_preds = score_baseline(
    seq['y_scaled'], split,
    cfg['sequence']['window_size'],
    len(X_test_t), target_scaler
)

print_metrics(lstm_metrics, 'LSTM')
print_metrics(base_metrics, 'Baseline')

## Step 5 — Save Checkpoint

In [ ]:
import pickle
ckpt_path = os.path.join('..', cfg['output']['checkpoint_path'])
os.makedirs(os.path.dirname(ckpt_path), exist_ok=True)

checkpoint = {
    'model_state'  : model.state_dict(),
    'input_size'   : len(FEATURE_COLS),
    'hidden_size'  : cfg['model']['hidden_size'],
    'window'       : cfg['sequence']['window_size'],
    'feature_cols' : FEATURE_COLS,
    'target_col'   : TARGET_COL,
    'feat_scaler'  : feat_scaler,
    'target_scaler': target_scaler,
    'metrics'      : lstm_metrics
}

with open(ckpt_path, 'wb') as f:
    pickle.dump(checkpoint, f)
print(f'saved → {ckpt_path}')